In [16]:
"""
Basic paser of Nasdaq ITCH-5.0 binary data
@author Richie R. Ma, ruhcuan2@illinois
"""

# Binary data processer using struct

import itch_message_templates
import pandas as pd
import struct
import gzip
# from itch_data_download import get_data
from tqdm import tqdm
import os
from pandas import isnull, notnull
from tabulate import tabulate
# %% message formats
# finding the start and end locations of each message type
# this is done by regular expression

"""
# binary encodes of messages
# System Event Message: S
# Stock directory: R
# Stock trading action: H
# Shortsale test: Y
# Market participation: L
# MWCB Decline Level Message: V
# MWCB Status Message: W
# Quoting Period Update: K
# Limit Up – Limit Down (LULD) Auction Collar: J
# Operational Halt: h
# Add Order – No MPID Attribution: A
#  Add Order with MPID Attribution: F
# Order ExecutedMessage: E
# Order Executed With PriceMessage: C
# Order Cancel Message: X
# Order Delete Message: D
# Order Replace message: U
# Trade Message (Non---Cross): P
# Cross trade message: Q
# Broken Trade / Order ExecutionMessage: B
# Net Order Imbalance Indicator (NOII)Message: I
# Direct Listing with Capital Raise Price Discovery Message: O
"""


def Nasdaq_parser(path, exchange, max_read_msgs=None, dataframe=True):

    if exchange not in ['Nasdaq', 'BX', 'PSX']:
        raise Exception(
            'Exchanges should be one of the following three: Nasdaq, BX, or PSX')

        # f.read() can directly read from the current point, no need to
        # explicitly specify the starting point
        # TODO: need to gove the file path
        # TODO: Need to use multiprocessing
        # TODO: This version only supports local file

    msgs_SystemEvent = []
    msgs_Stockdirectory = []
    msgs_tradingaction = []
    msgs_Shortsaletest = []
    msgs_MarketParticipant = []
    msgs_MWCBDeclineLevel = []
    msgs_MWCBStatus = []
    msgs_QuotingPeriodUpdate = []
    msgs_LULDAuctionCollar = []
    msgs_OperationHalt = []
    msgs_AddOrder = []
    msgs_AddOrderMPID = []
    msgs_ExecutedMessage = []
    msgs_ExecutedMessagewithPrice = []
    msgs_OrderCancel = []
    msgs_OrderDelete = []
    msgs_OrderReplace = []
    msgs_TradeMsgNocross = []
    msgs_TradeMsgCross = []
    msgs_BrokenTrdMsg = []
    msgs_NetOrdbal = []
    msgs_NetOrdbal = []
    msgs_Retailimprovement = []
    msgs_directlisting = []

    # initialize the progress bar
    # cutting the binary data to blocks
    # Remember: In .read(n) from a binary file,
    # n is the number of bytes to read starting from the current file pointer
    # it does NOT include the current position itself.
    # ----No msg field needed in msg function-----
    # TODO: check the length

    if isnull(max_read_msgs):
        print('maximum number of messages read does not provide. Read the whole file by default')
        max_read = os.path.getsize(path)
        print(f'Read total bytes: {max_read}')

    else:
        max_read = max_read_msgs
        print(f'Read maximum number of messages {max_read_msgs}')

    # we need to track the byte read location, I make the following function
    # not so helpful and necessary, might be deprecated later.
    """
    def byte_read(path, byte_size):
        
        with gzip.open(path, 'rb') as f:

            data = f.read(byte_size)
            byte_loc += len(data)
            
        if data:
            return data
        else:
            raise StopIteration('End of PCAP file reached!')
    """

    with open(path, 'rb') as f:

        msg_header = f.read(1)
        if isnull(max_read_msgs):
            read = 1
        else:
            read = 0

        # total should be set correctly, otherwise the progress bar can't be shown correctly
        # Directly set the bytes maximum read would not be exactly eqaul to what you set
        # I define the maximum number of messages read

        if notnull(max_read_msgs):
            unit = 'msgs'
        else:
            unit = 'bytes'

        with tqdm(total=max_read, desc="Reading", ncols=100, unit=unit) as pbar:

            while msg_header and read < max_read:

                # print(start)

                if msg_header == b'S':
                    msgs_blocks = f.read(11)
                    msgs_SystemEvent.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 11
                        pbar.update(11)

                elif msg_header == b'R':
                    msgs_blocks = f.read(38)
                    msgs_Stockdirectory.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 38
                        pbar.update(38)

                elif msg_header == b'H':
                    msgs_blocks = f.read(24)
                    msgs_tradingaction.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 24
                        pbar.update(24)

                elif msg_header == b'Y':
                    msgs_blocks = f.read(19)
                    msgs_Shortsaletest.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 19
                        pbar.update(19)

                elif msg_header == b'L':
                    msgs_blocks = f.read(25)
                    msgs_MarketParticipant.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 25
                        pbar.update(25)

                elif msg_header == b'V':
                    msgs_blocks = f.read(34)
                    msgs_MWCBDeclineLevel.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 34
                        pbar.update(34)

                elif msg_header == b'W':
                    msgs_blocks = f.read(11)
                    msgs_MWCBStatus.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 11
                        pbar.update(11)

                elif (msg_header == b'K') and (exchange == "Nasdaq"):
                    msgs_blocks = f.read(27)
                    msgs_QuotingPeriodUpdate.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 27
                        pbar.update(27)

                elif (msg_header == b'J') and (exchange == "Nasdaq"):
                    msgs_blocks = f.read(34)
                    msgs_LULDAuctionCollar.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 34
                        pbar.update(34)

                elif msg_header == b'h':
                    msgs_blocks = f.read(20)
                    msgs_OperationHalt.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 20
                        pbar.update(20)

                elif msg_header == b'A':
                    msgs_blocks = f.read(35)
                    msgs_AddOrder.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 35
                        pbar.update(35)

                elif msg_header == b'F':
                    msgs_blocks = f.read(39)
                    msgs_AddOrderMPID.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 39
                        pbar.update(39)

                elif msg_header == b'E':
                    msgs_blocks = f.read(30)
                    msgs_ExecutedMessage.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 30
                        pbar.update(30)

                elif msg_header == b'C':
                    msgs_blocks = f.read(35)
                    msgs_ExecutedMessagewithPrice.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 35
                        pbar.update(35)

                elif msg_header == b'X':
                    msgs_blocks = f.read(22)
                    msgs_OrderCancel.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 22
                        pbar.update(22)

                elif msg_header == b'D':
                    msgs_blocks = f.read(18)
                    msgs_OrderDelete.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 18
                        pbar.update(18)

                elif msg_header == b'U':
                    msgs_blocks = f.read(34)
                    msgs_OrderReplace.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 34
                        pbar.update(34)

                elif msg_header == b'P':
                    msgs_blocks = f.read(43)
                    msgs_TradeMsgNocross.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 43
                        pbar.update(43)

                elif msg_header == b'Q' and (exchange == "Nasdaq"):
                    msgs_blocks = f.read(39)
                    msgs_TradeMsgCross.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 39
                        pbar.update(39)

                elif msg_header == b'B':
                    msgs_blocks = f.read(18)
                    msgs_BrokenTrdMsg.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 18
                        pbar.update(18)

                elif msg_header == b'I' and (exchange == "Nasdaq"):
                    msgs_blocks = f.read(49)
                    msgs_NetOrdbal.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 49
                        pbar.update(49)

                elif msg_header == b'N' and (exchange == "BX"):
                    msgs_blocks = f.read(19)
                    msgs_Retailimprovement.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 19
                        pbar.update(19)

                elif msg_header == b'O' and (exchange == "Nasdaq"):
                    msgs_blocks = f.read(47)
                    msgs_directlisting.append(msgs_blocks)

                    if notnull(max_read_msgs):
                        read += 1
                        pbar.update(1)
                    else:
                        read += 47
                        pbar.update(47)

                if isnull(max_read_msgs):
                    pbar.update(1)
                    read += 1
                msg_header = f.read(1)

    msgs_SystemEvent = list(
        map(lambda item: itch_message_templates.SystemEvent(item), msgs_SystemEvent))
    msgs_Stockdirectory = list(
        map(lambda item: itch_message_templates.Stockdirectory(item), msgs_Stockdirectory))
    msgs_tradingaction = list(
        map(lambda item: itch_message_templates.tradingaction(item), msgs_tradingaction))
    msgs_Shortsaletest = list(
        map(lambda item: itch_message_templates.Shortsaletest(item), msgs_Shortsaletest))
    msgs_MarketParticipant = list(
        map(lambda item: itch_message_templates.MarketParticipant(item), msgs_MarketParticipant))
    msgs_MWCBDeclineLevel = list(
        map(lambda item: itch_message_templates.MWCBDeclineLevel(item), msgs_MWCBDeclineLevel))
    msgs_MWCBStatus = list(
        map(lambda item: itch_message_templates.MWCBStatus(item), msgs_MWCBStatus))
    msgs_QuotingPeriodUpdate = list(
        map(lambda item: itch_message_templates.QuotingPeriodUpdate(item), msgs_QuotingPeriodUpdate))
    msgs_LULDAuctionCollar = list(
        map(lambda item: itch_message_templates.LULDAuctionCollar(item), msgs_LULDAuctionCollar))
    msgs_OperationHalt = list(
        map(lambda item: itch_message_templates.OperationHalt(item), msgs_OperationHalt))
    msgs_AddOrder = list(
        map(lambda item: itch_message_templates.AddOrder(item), msgs_AddOrder))
    msgs_AddOrderMPID = list(
        map(lambda item: itch_message_templates.AddOrderMPID(item), msgs_AddOrderMPID))
    msgs_ExecutedMessage = list(
        map(lambda item: itch_message_templates.ExecutedMessage(item), msgs_ExecutedMessage))
    msgs_ExecutedMessagewithPrice = list(
        map(lambda item: itch_message_templates.ExecutedMessagewithPrice(item), msgs_ExecutedMessagewithPrice))
    msgs_OrderCancel = list(
        map(lambda item: itch_message_templates.OrderCancel(item), msgs_OrderCancel))
    msgs_OrderDelete = list(
        map(lambda item: itch_message_templates.OrderDelete(item), msgs_OrderDelete))
    msgs_OrderReplace = list(
        map(lambda item: itch_message_templates.OrderReplace(item), msgs_OrderReplace))
    msgs_TradeMsgNocross = list(
        map(lambda item: itch_message_templates.TradeMsgNocross(item), msgs_TradeMsgNocross))
    msgs_TradeMsgCross = list(
        map(lambda item: itch_message_templates.TradeMsgCross(item), msgs_TradeMsgCross))
    msgs_BrokenTrdMsg = list(
        map(lambda item: itch_message_templates.BrokenTrdMsg(item), msgs_BrokenTrdMsg))
    msgs_NetOrdbal = list(
        map(lambda item: itch_message_templates.NetOrdbal(item), msgs_NetOrdbal))
    msgs_Retailimprovement = list(
        map(lambda item: itch_message_templates.Retailimprovement(item), msgs_Retailimprovement))
    msgs_directlisting = list(
        map(lambda item: itch_message_templates.directlisting(item), msgs_directlisting))

    results = {'System_Event': msgs_SystemEvent,
               'Stockdirectory': msgs_Stockdirectory,
               'tradingaction': msgs_tradingaction,
               'Shortsaletest': msgs_Shortsaletest,
               'MarketParticipant': msgs_MarketParticipant,
               'MWCBDeclineLevel': msgs_MWCBDeclineLevel,
               'MWCBStatus': msgs_MWCBStatus,
               'QuotingPeriodUpdate': msgs_QuotingPeriodUpdate,
               'LULDAuctionCollar': msgs_LULDAuctionCollar,
               'OperationHalt': msgs_OperationHalt,
               'AddOrder': msgs_AddOrder,
               'AddOrderMPID': msgs_AddOrderMPID,
               'ExecutedMessage': msgs_ExecutedMessage,
               'ExecutedMessagewithPrice': msgs_ExecutedMessagewithPrice,
               'OrderCancel': msgs_OrderCancel,
               'OrderDelete': msgs_OrderDelete,
               'OrderReplace': msgs_OrderReplace,
               'TradeMsgNocross': msgs_TradeMsgNocross,
               'TradeMsgCross': msgs_TradeMsgCross,
               'BrokenTrdMsg': msgs_BrokenTrdMsg,
               'NetOrdbal': msgs_NetOrdbal,
               'directlisting': msgs_directlisting
               }

    results_info = pd.DataFrame({'msg_types': list(results.keys()), 'number_msgs': list(
        map(lambda item: len(item), results.values()))})

    print(tabulate(results_info, headers=[
          'msg_types', 'number_msgs'], tablefmt="grid"))

    if dataframe == True:

        def timestamps_conversion(df_msgs):
            # note that ITCH does not provide the date field
            # this could be done from the path
            # e.g., 12302019.NASDAQ_ITCH50.gz, the date ranges from -16 to -23

            if not isinstance(df_msgs, pd.DataFrame):
                raise Exception(
                    'Timestamps conversion should be based on dataframe')

            if 'Time' in df_msgs.columns:

                date = pd.Timestamp(
                    year=int(path[-19:-15]), month=int(path[-15:-13]), day=int(path[-13:-11]))
                df_msgs['Time'] = date + \
                    pd.to_timedelta(df_msgs['Time'], unit='ns')
                df_msgs['Time'] = df_msgs['Time'].dt.tz_localize(
                    'America/New_York')

                return df_msgs

            else:
                return pd.DataFrame()

        results = dict(
            map(lambda item: (item[0], timestamps_conversion(pd.DataFrame(item[1]))), results.items()))

    return results

In [17]:
# Takes pcap file path and returns a file path to the extracted raw ITCH data

from scapy.all import rdpcap, UDP, Raw

def pcap_to_itch(pcap_fpath):
    pkts =  rdpcap(pcap_fpath)
    raw_fpath = 'data/itch/raw_itch_data_' + pcap_fpath[-20:-5] + '.bin' 
    with open(raw_fpath, 'wb') as out:
        for pkt in pkts:
            if UDP in pkt:
                payload = bytes(pkt[UDP].payload)

                # Skip packets too small to hold MoldUDP64 header
                if len(payload) < 20:
                    continue

                # MoldUDP header
                session_id = payload[0:10]       # ASCII
                seq_num    = payload[10:18]      # uint64
                msg_count  = int.from_bytes(payload[18:20], 'big')

                offset = 20
                for _ in range(msg_count):
                    # Each message: [length][message]
                    if offset + 2 > len(payload):
                        break
                    msg_len = int.from_bytes(payload[offset:offset+2], 'big')
                    offset += 2
                    msg = payload[offset:offset+msg_len]
                    offset += msg_len

                    # Write ONLY the ITCH message body
                    out.write(msg)
                    
    return raw_fpath

In [18]:
pcap_fpath = 'data/pcap/ny4-xnas-tvitch-a-20250201T010000.pcap'
raw_fpath = pcap_to_itch(pcap_fpath)
test = Nasdaq_parser(
    path=raw_fpath, exchange='Nasdaq',
    max_read_msgs=None, dataframe=False)

maximum number of messages read does not provide. Read the whole file by default
Read total bytes: 3641555


Reading: 100%|███████████████████████████████████| 3641555/3641555 [00:00<00:00, 13889732.43bytes/s]


+----+--------------------------+---------------+
|    | msg_types                |   number_msgs |
+====+==========================+===============+
|  0 | System_Event             |             2 |
+----+--------------------------+---------------+
|  1 | Stockdirectory           |             0 |
+----+--------------------------+---------------+
|  2 | tradingaction            |             0 |
+----+--------------------------+---------------+
|  3 | Shortsaletest            |             0 |
+----+--------------------------+---------------+
|  4 | MarketParticipant        |             0 |
+----+--------------------------+---------------+
|  5 | MWCBDeclineLevel         |             0 |
+----+--------------------------+---------------+
|  6 | MWCBStatus               |             0 |
+----+--------------------------+---------------+
|  7 | QuotingPeriodUpdate      |             0 |
+----+--------------------------+---------------+
|  8 | LULDAuctionCollar        |             0 |
